In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
    summarize_index_overlap,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    SpenderID,
    common_translate,
    split_data,
    collapse_col,
    find_redundant_cols,
)

### Target Population Filtering

The donors in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty columns. The duplicate column `dr_split_2` was kept, as it is common to have that information as well.

In [ ]:
data = pd.read_parquet(input_data)
donors = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_id_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
targetpop = pd.read_parquet(targetpop_data)
data = data[donors.isin(targetpop["donor_et_id_et"])]
display(
    Markdown(
        f"""The filter process reduced the number of donors in the data ({donors.nunique()}) and target population ({targetpop["donor_et_id_et"].nunique()})
            to {donors[donors.isin(targetpop["donor_et_id_et"])].nunique()} in the processed data.
        """
    )
)

In [ ]:
data = drop_col_few_distinct(data)
# data = drop_duplicate_columns(data) removes dr_split_2

### Integration of Seperated Institute Data

In this file, the {term}`DSO` and {term}`IQTIG` data is not connected (see [](general:ic)). 

In [ ]:
idcols = ["donor_et_dso", "donor_et_id_et"]
data = split_data(data, idcols)
assert len(data) == 2, "Not 2 different row types present!?"

Both sources contribute HLA testing data, however the {term}`DSO` splits those into their respective groups, while {term}`ET` combines the results. The following analysis shows the overlap. 

In [ ]:
summarize_index_overlap(data["donor_et_id_et"], data["donor_et_dso"], "ET", "DSO")

We dropped the {term}`DSO` data, as there are only a few donors, who only occur in their data and the automatic consolidation would be too error prone. 

In [ ]:
data = data["donor_et_id_et"].reset_index()

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

There is no column differentiating between different types of tests (see [](general:rf)). We kept all rows.

In [ ]:
display_long_data_doc(
    data,
    [
        "donor_et_id_et",
    ],
    "enter_date",
    None,
)

### Unit Conversions

We applied the common translations (see [](general:uc)).

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

### Consolidating Columns

No consolidation was necessary, as the dataset only contained {term}`ET` data after filtering (see [](general:crc)).

In [ ]:
red = find_redundant_cols(data)
assert len(red) == 0

## Intermediate Dataset

In [ ]:
indcols = ["donor_et_id_et"]
data = data.sort_index(axis=1).sort_values(indcols + ["enter_date"], axis=0)
data = data.set_index(indcols)

In [ ]:
# Another base class might be necessary, see util.py
# describe columns, without checks for now, order is important
class DonorPostmortemLabHLA(SpenderID):
    antigens: Series[str] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="HLA Phenotype Antigens",
        description="The HLA antigens detected by a lab test for this donor",
    )
    enter_date: Series[float] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Enter data",
        description="When was this test result entered into the database?",
    )
    sampling_tissue_imprecise: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Sampled Tissue",
        description="Where was this tissue taken?",
        isin=["Peripheral blood", "XX_Unknown", "Spleen", "Lymph nodes", "Other"],
    )
    sso_ssp: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Typing Method",
        description="Was SSP or SSO used for typing?",
        isin=["SSP", "SSO"],
    )

    class Config:
        title = "Donor Postmortem HLA Dataset"
        description = "Each row represents an HLA typing result. The data is based on the 'element_spender_postmortem_labor_hla.csv' file. It contains data from the ET and DSO."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(DonorPostmortemLabHLA, data)

In [ ]:
DonorPostmortemLabHLA.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    DonorPostmortemLabHLA.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)